# Neo4j를 활용한 GraphRAG (Graph Retrieval-Augmented Generation)

## 1. RAG (Retrieval-Augmented Generation)란?

### 1.1 RAG의 필요성

**문제점**: 대규모 언어모델(LLM)의 한계
- ❌ **지식의 한계**: 학습 데이터 시점까지의 정보만 알고 있음
- ❌ **환각(Hallucination)**: 없는 정보를 만들어내는 경향
- ❌ **도메인 특화 지식 부족**: 특정 기업/조직의 내부 정보 모름
- ❌ **업데이트 어려움**: 새로운 정보를 반영하려면 재학습 필요

**해결책**: RAG (Retrieval-Augmented Generation)
- ✅ **최신 정보 제공**: 외부 데이터베이스에서 관련 정보 검색
- ✅ **정확성 향상**: 검색된 실제 데이터 기반 답변 생성
- ✅ **출처 제공**: 답변의 근거를 명확히 제시
- ✅ **비용 효율적**: 전체 모델 재학습 없이 정보 업데이트


### 1.2 RAG의 기본 구조

RAG는 크게 3단계로 구성됩니다:

```
┌─────────────┐
│  1. Index   │  문서를 벡터로 변환하여 DB에 저장
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 2. Retrieve │  질문과 유사한 문서를 검색
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 3. Generate │  LLM이 검색된 문서를 참고하여 답변 생성
└─────────────┘
```

**단계별 상세**:

1. **Index (색인)**: 
   - 문서를 작은 청크(chunk)로 분할
   - 임베딩 모델로 벡터화
   - 데이터베이스에 저장

2. **Retrieve (검색)**:
   - 사용자 질문을 벡터화
   - 유사도 기반으로 관련 문서 검색
   - Top-K 개의 가장 관련있는 문서 추출

3. **Generate (생성)**:
   - 검색된 문서를 컨텍스트로 제공
   - LLM이 질문 + 컨텍스트를 받아 답변 생성
   - 근거가 있는 정확한 답변 제공


## 2. VectorDB RAG vs GraphDB RAG 비교

### 2.1 Vector RAG (전통적인 RAG)

**데이터 저장 방식**: 벡터 데이터베이스 (예: Pinecone, Chroma, FAISS)

```
문서 A → [0.2, 0.8, 0.1, ...] ───┐
문서 B → [0.5, 0.3, 0.9, ...] ───┤ 벡터 DB
문서 C → [0.1, 0.7, 0.4, ...] ───┘
```

**장점** ✅:
- 의미적 유사도 검색에 강함
- 간단한 구조로 빠른 구현 가능
- 대용량 데이터 처리에 효율적
- 임베딩 기반 검색 성능 우수

**단점** ❌:
- **관계 정보 손실**: 엔티티 간 연결 관계를 파악하기 어려움
- **맥락 파악 한계**: "누가 누구와 어떤 관계인가?" 같은 질문에 약함
- **복잡한 추론 어려움**: 여러 단계의 관계 추론이 필요한 질문에 한계
- **중복 정보**: 같은 엔티티가 여러 문서에 분산되어 저장

**적합한 사용 사례**:
- 단순 키워드/의미 검색
- FAQ 시스템
- 문서 유사도 검색


### 2.2 Graph RAG (그래프 기반 RAG)

**데이터 저장 방식**: 그래프 데이터베이스 (예: Neo4j, Amazon Neptune)

```
    (기자A)──[WORKS_FOR]──→(언론사X)
       │                         ↑
       │                         │
   [WRITTEN_BY]            [PUBLISHED_BY]
       │                         │
       ↓                         │
    (뉴스1)──[BELONGS_TO]──→(카테고리)
       │
   [PUBLISHED_IN]
       │
       ↓
   (2025-11)
```

**장점** ✅:
- **관계 보존**: 엔티티 간의 연결 관계를 명시적으로 저장
- **복잡한 질의**: "이 기자가 작성한 경제 뉴스는?" 같은 복잡한 질문 처리
- **맥락 이해**: 그래프 경로를 따라가며 풍부한 맥락 제공
- **추론 가능**: 여러 홉(hop)을 거쳐 관계 추론
- **중복 제거**: 같은 엔티티는 하나의 노드로 통합

**단점** ❌:
- 초기 그래프 구축이 복잡함
- 벡터 검색보다 구현이 어려움
- 그래프 설계가 성능에 큰 영향

**적합한 사용 사례**:
- 지식 그래프 기반 QA
- 복잡한 관계 분석 (소셜 네트워크, 추천 시스템)
- 엔티티 중심의 검색 (특정 인물, 기업, 제품)
- 다단계 추론이 필요한 질문


### 2.3 비교표

| 항목 | Vector RAG | Graph RAG |
|------|-----------|-----------|
| **데이터 구조** | 벡터 (Embeddings) | 노드 + 관계 (Graph) |
| **검색 방식** | 코사인 유사도 | Cypher 쿼리 (경로 탐색) |
| **관계 표현** | ❌ 어려움 | ✅ 명시적 |
| **구현 난이도** | ⭐⭐ 쉬움 | ⭐⭐⭐⭐ 어려움 |
| **복잡한 질의** | ❌ 한계 있음 | ✅ 강력함 |
| **확장성** | ✅ 우수 | ⚠️ 그래프 크기에 따라 다름 |
| **추론 능력** | ❌ 제한적 | ✅ 강력함 |
| **유지보수** | ⭐⭐ 쉬움 | ⭐⭐⭐ 보통 |


## 3. Neo4j를 활용한 GraphRAG 실습 준비작업



### 3.1 API Key 설정
- [OpenAI API Key 발급](https://platform.openai.com/api-keys)


In [1]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


True

### 3.2 Neo4j 연결 클래스

In [2]:
class Singleton(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super(Singleton, cls).__call__(*args, **kwargs)
        return cls._instances[cls]

In [3]:
# Neo4j 연결 클래스 (Singleton 패턴)
from neo4j import GraphDatabase

class Neo4jConnection(metaclass=Singleton):
    """Neo4j 데이터베이스 연결 및 작업을 위한 클래스"""
    
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("Neo4j 연결 성공!")
    
    def close(self):
        if self.driver is not None:
            self.driver.close()
            print("Neo4j 연결 종료")
    
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]


### 3.3 Neo4j 연결 및 데이터 확인 

In [10]:
# Neo4j 연결
neo4j_conn = Neo4jConnection(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="test1234"
)


In [11]:
# 저장된 데이터 확인
query = """
MATCH (n)
RETURN labels(n)[0] as NodeType, count(n) as Count
ORDER BY Count DESC
"""

results = neo4j_conn.execute_query(query)
print("Neo4j 데이터 현황:\n")
for record in results:
    print(f"  {record['NodeType']:<15} : {record['Count']:>3}개")


Neo4j 데이터 현황:

  Reporter        :  20개
  News            :  20개
  Publisher       :  15개
  Category        :   4개
  YearMonth       :   1개


## 4. GraphRAG 시스템 구축

### 4.1 질문 유형별 Cypher 쿼리 템플릿

GraphRAG의 핵심은 **자연어 질문을 Cypher 쿼리로 변환**하는 것입니다!


In [ ]:
from enum import Enum

class CypherQueryTemplates(Enum):
    """다양한 질문 유형을 Enum으로 관리하는 Cypher 템플릿"""

    NEWS_BY_CATEGORY = "news_by_category"
    NEWS_BY_PUBLISHER = "news_by_publisher"
    NEWS_BY_REPORTER = "news_by_reporter"

    def build(self, **kwargs):
        """템플릿 유형에 따라 Cypher 쿼리 생성"""
        if self is CypherQueryTemplates.NEWS_BY_CATEGORY:
            category_name = kwargs["category_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:BELONGS_TO]->(c:Category {{name: "{category_name}"}})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_PUBLISHER:
            publisher_name = kwargs["publisher_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {{name: "{publisher_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_REPORTER:
            reporter_name = kwargs["reporter_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:WRITTEN_BY]->(r:Reporter {{name: "{reporter_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        raise ValueError(f"지원하지 않는 템플릿: {self}")


#### 테스트 

##### 카테고리로 뉴스 검색

In [ ]:
# 카테고리로 뉴스 검색
query = CypherQueryTemplates.NEWS_BY_CATEGORY.build(category_name="경제", limit_no=3)
print(query)


            MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: "경제"})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 3
            


In [49]:
results = neo4j_conn.execute_query(query)

In [50]:
results

[<Record 제목='1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑' 내용="10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%, 전년比 1.7%p↑…5년 평균과 유사 ⓒ News1 윤주희 디자이너 (세종=뉴스1) 임용우 기자 = 올해 1~10월 국세 수입이 지난해 같은 기간보다 37조 1000억 원 늘어난 것으로 집계됐다. 지난해와 올해 상반기 기업 실적 개선, 성과급 지급 확대 등이 이어지며 법인세와 소득세가 크게 증가한 영향이다. 기획재정부가 28일 발표한 '2025년 10월 국세수입 현황'에 따르면 10월 국세 수입은 41조 1000억 원으로 전년 동월(38조 3000억 원)보다 2조 8000억 원 증가했다. 법인세는 상반기 기업 실적 개선에 따른 중소기업 중간예납 분납분 증가, 이자·배당 원천소득 증가 등으로 7000억 원 늘었다. 소득세는 근로자 수 증가와 총급여 지급액 확대에 따른 근로소득세 증가로 9000억 원 증가했다. 부가가치세는 올해 2기 예정신고분 납부 증가와 환율 상승 영향 등으로 7000억 원 증" 언론사='뉴스1' 기자='임용우 기자' 발행일='2025-11-28 11:00:00' 뉴스링크='https://n.news.naver.com/mnews/article/421/0008630778'>,
 <Record 제목='지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개' 내용="주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 자율주행자동차 사고통계 공개 누리집 모습.(한국교통안전공단 제공)뉴스1ⓒ news1 (서울=뉴스1) 김동규 기자 = 한국교통안전공단 자동차안전연구원(TS)이 정부와 함께 국내 자율주행차 사고통계를 처음으로 공개한다. 2022년 이후 사고가 매년 증가하는 가운데 자율주행모드 운행 중 사고가 전체의 35%에 이르는 것으로 나타나, 안전성 확보를 위한 체계적 데이터 공개가 본격화됐다는 평가다. TS 자

In [51]:
for record in results:
    print(f"제목: {record['제목']}")
    print(f"내용: {record['내용'][:50]}...")
    print(f"언론사: {record['언론사']}")
    print(f"뉴스링크: {record['뉴스링크']}")
    break 

제목: 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
내용: 10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%,...
언론사: 뉴스1
뉴스링크: https://n.news.naver.com/mnews/article/421/0008630778


##### 기자 통계 조회

In [37]:
# 기자 통계 조회
query = CypherQueryTemplates.REPORTER_STATISTICS.build()
print(query)


            MATCH (r:Reporter)-[:WRITTEN_BY]-(n:News)
            MATCH (r)-[:WORKS_FOR]->(p:Publisher)
            WITH r.name as 기자명, p.name as 소속언론사, count(n) as 작성기사수
            RETURN 기자명, 소속언론사, 작성기사수
            ORDER BY 작성기사수 DESC
            LIMIT 10
            


In [38]:
results = neo4j_conn.execute_query(query)

In [39]:
results

[<Record 기자명='장훈경 기자' 소속언론사='SBS' 작성기사수=1>,
 <Record 기자명='정남구 기자' 소속언론사='한겨레' 작성기사수=1>,
 <Record 기자명='신기림 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김성식 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='임용우 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김동규 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김종윤 기자' 소속언론사='SBS Biz' 작성기사수=1>,
 <Record 기자명='김지헌 기자' 소속언론사='헤럴드경제' 작성기사수=1>,
 <Record 기자명='함영훈 기자' 소속언론사='헤럴드경제' 작성기사수=1>,
 <Record 기자명='이광수 기자' 소속언론사='국민일보' 작성기사수=1>]

In [40]:
for record in results:
    print(f"기자명: {record['기자명']}")
    print(f"소속언론사: {record['소속언론사']}")
    print(f"작성기사수: {record['작성기사수']}")
    break 

기자명: 장훈경 기자
소속언론사: SBS
작성기사수: 1


##### 키워드 검색

In [41]:
# 키워드 검색
query = CypherQueryTemplates.SEARCH_BY_KEYWORD.build(keyword="자율주행")
print(query)


            MATCH (n:News)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            WHERE n.title CONTAINS "자율주행" OR n.content CONTAINS "자율주행"
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, r.name as 기자,
                   n.link as 뉴스링크
            LIMIT 5
            


In [42]:
results = neo4j_conn.execute_query(query)

In [43]:
results

[<Record 제목='지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개' 내용="주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 자율주행자동차 사고통계 공개 누리집 모습.(한국교통안전공단 제공)뉴스1ⓒ news1 (서울=뉴스1) 김동규 기자 = 한국교통안전공단 자동차안전연구원(TS)이 정부와 함께 국내 자율주행차 사고통계를 처음으로 공개한다. 2022년 이후 사고가 매년 증가하는 가운데 자율주행모드 운행 중 사고가 전체의 35%에 이르는 것으로 나타나, 안전성 확보를 위한 체계적 데이터 공개가 본격화됐다는 평가다. TS 자동차안전연구원은 28일 국토교통부와 함께 자율주행차 사고 정보를 정례적으로 공개한다고 밝혔다. 제한적으로 관리되던 사고 데이터를 투명하게 공개하는 것은 이번이 처음으로, 자율주행차 상용화를 앞두고 국민 신뢰도를 높이기 위한 조치다. 사고통계는 '자율주행자동차 사고조사위원회' 누리집을 통해 확인할 수 있다. TS는 제작사로부터 제출받는 자율주행차 교통사고 신고서를 바탕으로 △연도 △운행모드 △시간대 △날씨" 카테고리='경제' 언론사='뉴스1' 기자='김동규 기자' 뉴스링크='https://n.news.naver.com/mnews/article/421/0008631187'>]

In [45]:
for record in results:
    print(f"제목: {record['제목']}")
    print(f"내용: {record['내용'][:50]}...")
    print(f"언론사: {record['언론사']}")
    print(f"뉴스링크: {record['뉴스링크']}")
    break 

제목: 지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개
내용: 주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 ...
언론사: 뉴스1
뉴스링크: https://n.news.naver.com/mnews/article/421/0008631187


### 4.2 LangChain Tool 정의

이제 CypherQueryTemplates를 LangChain의 Tool로 변환하여 Agent가 사용할 수 있도록 만들겠습니다.


#### search_news_by_category

In [ ]:
from langchain_core.tools import tool

@tool
def search_news_by_category(category_name:str, limit_no:int=2) -> str:
    """
    category_name에 속하는 뉴스 기사 limit_no개를 검색
    """
    query = CypherQueryTemplates.NEWS_BY_CATEGORY.build(
        category_name=category_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[카테고리: {category_name}]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {record['내용']}
            - 언론사: {answer['언론사']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {record['뉴스링크']}
        """

    return result

#### search_news_by_publisher

In [ ]:
@tool
def search_news_by_publisher(publisher_name:str, limit_no:int=2) -> str:
    """
    publisher_name(언론사)가 발생한 뉴스 기사 limit_no개를 검색
    """
    query = CypherQueryTemplates.NEWS_BY_PUBLISHER.build(
        publisher_name=publisher_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[언론사: {publisher_name}]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {record['내용']}
            - 카테고리: {answer['카테고리']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {record['뉴스링크']}
        """

    return result

#### search_news_by_reporter

In [ ]:
@tool
def search_news_by_reporter(reporter_name:str, limit_no:int=2) -> str:
    """
    reporter_name(기자명)가 작성한 뉴스 기사 limit_no개를 검색
    """
    query = CypherQueryTemplates.NEWS_BY_REPORTER.build(
        reporter_name=reporter_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[기자명: {reporter_name}]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {record['내용']}
            - 카테고리: {answer['카테고리']}
            - 언론사: {answer['언론사']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {record['뉴스링크']}
        """

    return result

#### Text2CypherRetriever

In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG

@tool
def search_by_text2cypher(question:str, limit_no:int=2) -> str:
    """
    
    """
    # LLM 설정
    llm = OpenAILLM(
        model_name="gpt-5-nano"
    )

    # Neo4j 스키마 정의 (우리의 뉴스 데이터 구조)
    neo4j_schema = """
    노드 속성:
    News {title: STRING, content: STRING, link: STRING, publishDate: STRING}
    Category {name: STRING}  # 예: "경제", "IT/과학", "생활/문화", "세계"
    Publisher {name: STRING}  # 언론사명
    Reporter {name: STRING}   # 기자명
    YearMonth {period: STRING}  # 예: "2025-11"

    관계:
    (:News)-[:BELONGS_TO]->(:Category)
    (:News)-[:PUBLISHED_BY]->(:Publisher)
    (:News)-[:WRITTEN_BY]->(:Reporter)
    (:News)-[:PUBLISHED_IN]->(:YearMonth)
    (:Reporter)-[:WORKS_FOR]->(:Publisher)
    """

    # (선택사항) 예시 쿼리 제공 - LLM이 더 정확한 Cypher를 생성하도록 도움
    examples = [
        "사용자 질문: 'IT/과학 카테고리의 뉴스를 보여줘' "
        "쿼리: MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: 'IT/과학'}) "
        "RETURN n.title LIMIT 5",
        
        "사용자 질문: '뉴스1에서 발행한 뉴스는?' "
        "쿼리: MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {name: '뉴스1'}) "
        "RETURN n.title LIMIT 5"
    ]

    # Text2CypherRetriever 초기화
    retriever = Text2CypherRetriever(
        driver=neo4j_conn.driver,  # 이미 연결된 Neo4j driver 사용
        llm=llm,
        neo4j_schema=neo4j_schema,
        examples=examples
    )

    answers = retriever.search(query_text=question)


    result = f"[기자명: {reporter_name}]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {record['내용']}
            - 카테고리: {answer['카테고리']}
            - 언론사: {answer['언론사']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {record['뉴스링크']}
        """

    return result

## 5. ReAct Agent with LangGraph

### 5.1 ReAct (Reasoning + Acting)란?

**ReAct**는 LLM이 **추론(Reasoning)**과 **행동(Acting)**을 결합하여 문제를 해결하는 패러다임입니다.

```
사용자 질문: "경제 카테고리의 뉴스를 찾아줘"
     ↓
┌────────────────────────────────────────────────┐
│  Thought (추론): 경제 카테고리 뉴스를 찾으려면  │
│  NEWS_BY_CATEGORY 템플릿을 사용해야겠다        │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Action (행동): execute_cypher_template 호출   │
│  - template_name: "NEWS_BY_CATEGORY"           │
│  - category_name: "경제"                       │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Observation (관찰): Neo4j에서 5건의 뉴스 검색 │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Thought (추론): 검색 결과를 사용자에게         │
│  이해하기 쉽게 정리해서 답변하자                │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Final Answer: "경제 카테고리 뉴스 5건을        │
│  찾았습니다. 1. ..."                           │
└────────────────────────────────────────────────┘
```

**핵심 특징**:
- **Thought**: LLM이 다음에 무엇을 해야 할지 추론
- **Action**: 도구(Tool)를 선택하고 실행
- **Observation**: 도구 실행 결과 확인
- **반복**: 필요시 Thought → Action → Observation 반복
- **Answer**: 최종 답변 생성


### 5.2 LangGraph ReAct Agent 구축

LangGraph의 `create_react_agent`를 사용하여 ReAct Agent를 생성합니다.


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# 1. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. System Prompt 정의
system_prompt = """당신은 Neo4j 그래프 데이터베이스에 저장된 뉴스 기사를 검색하는 전문 AI 어시스턴트입니다.

## 역할
사용자의 질문을 분석하여 적절한 Cypher 쿼리 템플릿을 선택하고 실행하여 뉴스 기사를 검색합니다.

## 사용 가능한 Cypher 쿼리 템플릿
1. **NEWS_BY_CATEGORY**: 특정 카테고리의 뉴스 검색 (예: 경제, 사회, IT과학, 생활문화)
2. **NEWS_BY_PUBLISHER**: 특정 언론사의 뉴스 검색 (예: 뉴스1, SBS, 국민일보)
3. **NEWS_BY_REPORTER**: 특정 기자가 작성한 뉴스 검색
4. **NEWS_BY_PUBLISHER_AND_CATEGORY**: 특정 언론사의 특정 카테고리 뉴스 검색
5. **REPORTER_STATISTICS**: 기자별 작성 기사 통계 조회
6. **SEARCH_BY_KEYWORD**: 키워드로 뉴스 제목 및 내용 검색

## 중요 지침
- 사용자 질문을 정확히 분석하여 가장 적합한 템플릿을 선택하세요.
- 템플릿 이름은 정확히 위의 이름을 사용해야 합니다 (대소문자 구분).
- 필요한 파라미터를 모두 추출하여 전달하세요.
- Tool 실행 결과를 사용자에게 친절하고 자연스럽게 설명하세요.
- 검색 결과가 없으면 사용자에게 다른 검색어를 제안하세요.

## 응답 스타일
- 친절하고 전문적인 톤을 유지하세요.
- 검색 결과를 명확하고 구조화된 형태로 제시하세요.
- 필요시 추가 정보나 관련 질문을 제안하세요.
"""

# 3. ReAct Agent 생성
agent = create_react_agent(
    llm,
    tools=tools,
    state_modifier=system_prompt
)

print("✅ ReAct Agent 생성 완료!")


### 5.3 Agent 실행 헬퍼 함수


In [ ]:
def run_agent(question: str, verbose: bool = True):
    """
    ReAct Agent를 실행하고 결과를 출력합니다.
    
    Args:
        question: 사용자 질문
        verbose: 상세 출력 여부 (True: 전체 과정 출력, False: 최종 답변만 출력)
    """
    print("="*80)
    print(f"🙋 질문: {question}")
    print("="*80)
    
    # Agent 실행
    result = agent.invoke({"messages": [("user", question)]})
    
    if verbose:
        print("\n📋 전체 실행 과정:\n")
        for i, message in enumerate(result["messages"], 1):
            print(f"\n--- Step {i} ---")
            
            # 사용자 메시지
            if hasattr(message, "type") and message.type == "human":
                print(f"👤 User: {message.content}")
            
            # AI 메시지 (Thought + Action)
            elif hasattr(message, "type") and message.type == "ai":
                if hasattr(message, "tool_calls") and message.tool_calls:
                    print(f"🤔 AI Thought & Action:")
                    if message.content:
                        print(f"   Thought: {message.content}")
                    for tool_call in message.tool_calls:
                        print(f"   Action: {tool_call['name']}")
                        print(f"   Arguments: {tool_call['args']}")
                else:
                    print(f"🤖 AI Final Answer:")
                    print(f"   {message.content}")
            
            # Tool 실행 결과 (Observation)
            elif hasattr(message, "type") and message.type == "tool":
                print(f"👁️ Observation (Tool Result):")
                # 결과가 너무 길면 처음 500자만 출력
                content = message.content
                if len(content) > 500:
                    print(f"   {content[:500]}...")
                    print(f"   ... (총 {len(content)}자)")
                else:
                    print(f"   {content}")
    
    print("\n" + "="*80)
    print("🎯 최종 답변:")
    print("="*80)
    
    # 최종 답변 추출
    final_message = result["messages"][-1]
    print(final_message.content)
    print("="*80 + "\n")
    
    return result


## 6. GraphRAG Agent 테스트

### 6.1 테스트 1: 카테고리별 뉴스 검색


In [ ]:
run_agent("경제 카테고리의 뉴스를 찾아줘")


### 6.2 테스트 2: 키워드 검색


In [ ]:
run_agent("자율주행에 관한 뉴스가 있어?")


### 6.3 테스트 3: 언론사별 검색


In [ ]:
run_agent("뉴스1에서 발행한 기사를 보여줘")


### 6.4 테스트 4: 기자 통계 조회


In [ ]:
run_agent("기자별로 작성한 기사 수 통계를 보여줘")


### 6.5 테스트 5: Tool 없이 일반 대화 (Agent의 판단력 테스트)


In [ ]:
run_agent("GraphRAG가 뭐야?")


## 7. 정리 및 요약

### 7.1 우리가 구축한 GraphRAG 시스템의 구조

```
┌─────────────────────────────────────────────────────────────┐
│                     GraphRAG 시스템                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  1️⃣ Data Layer (Neo4j Graph Database)                       │
│     ┌──────────────────────────────────────────┐            │
│     │  Nodes: News, Reporter, Publisher,        │            │
│     │         Category, YearMonth               │            │
│     │  Relationships: WRITTEN_BY, PUBLISHED_BY, │            │
│     │                 BELONGS_TO, PUBLISHED_IN, │            │
│     │                 WORKS_FOR                 │            │
│     └──────────────────────────────────────────┘            │
│                        ↓                                     │
│  2️⃣ Query Template Layer (CypherQueryTemplates)             │
│     ┌──────────────────────────────────────────┐            │
│     │  - NEWS_BY_CATEGORY                      │            │
│     │  - NEWS_BY_PUBLISHER                     │            │
│     │  - NEWS_BY_REPORTER                      │            │
│     │  - NEWS_BY_PUBLISHER_AND_CATEGORY        │            │
│     │  - REPORTER_STATISTICS                   │            │
│     │  - SEARCH_BY_KEYWORD                     │            │
│     └──────────────────────────────────────────┘            │
│                        ↓                                     │
│  3️⃣ Tool Layer (LangChain Tool)                             │
│     ┌──────────────────────────────────────────┐            │
│     │  execute_cypher_template()               │            │
│     │  - 템플릿 선택 및 파라미터 전달           │            │
│     │  - Cypher 쿼리 실행                      │            │
│     │  - 결과 포맷팅 및 반환                    │            │
│     └──────────────────────────────────────────┘            │
│                        ↓                                     │
│  4️⃣ Agent Layer (LangGraph ReAct Agent)                     │
│     ┌──────────────────────────────────────────┐            │
│     │  ReAct Cycle:                            │            │
│     │  1. Thought: 질문 분석 및 전략 수립       │            │
│     │  2. Action: Tool 선택 및 실행            │            │
│     │  3. Observation: 결과 확인               │            │
│     │  4. (반복 또는 최종 답변)                 │            │
│     └──────────────────────────────────────────┘            │
│                        ↓                                     │
│  5️⃣ LLM Layer (OpenAI GPT-4o-mini)                          │
│     ┌──────────────────────────────────────────┐            │
│     │  - 자연어 이해                            │            │
│     │  - 추론 및 의사결정                       │            │
│     │  - 답변 생성                              │            │
│     └──────────────────────────────────────────┘            │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```


### 7.2 핵심 개념 정리

#### 1. CypherQueryTemplates (Enum 기반 쿼리 관리)
- **목적**: Cypher 쿼리를 체계적으로 관리
- **장점**: 
  - 재사용성: 동일한 패턴의 쿼리를 템플릿화
  - 유지보수성: 쿼리 수정 시 한 곳만 변경
  - 타입 안정성: Enum으로 템플릿 이름 관리

#### 2. LangChain Tool
- **목적**: LLM이 Neo4j를 사용할 수 있도록 인터페이스 제공
- **핵심 요소**:
  - `@tool` 데코레이터: 함수를 LLM이 사용 가능한 도구로 변환
  - `args_schema`: Pydantic 모델로 입력 파라미터 정의
  - `description`: LLM이 도구 사용 시기를 판단하는 핵심 정보

#### 3. ReAct Pattern (Reasoning + Acting)
- **Thought**: LLM이 문제 해결 전략 수립
- **Action**: 선택한 도구 실행
- **Observation**: 도구 실행 결과 확인
- **반복**: 필요시 추가 도구 실행
- **Answer**: 최종 답변 생성

#### 4. LangGraph
- **목적**: 복잡한 AI 워크플로우를 그래프로 표현
- **create_react_agent**: ReAct 패턴을 자동으로 구현
- **state_modifier**: Agent의 시스템 프롬프트 설정
- **장점**: 
  - 상태 관리: 대화 히스토리 자동 관리
  - 유연한 워크플로우: 조건부 실행 흐름 구현
  - 디버깅 용이: 각 단계별 결과 확인 가능


### 7.3 Vector RAG vs GraphRAG 비교

| 측면 | Vector RAG | GraphRAG (우리 시스템) |
|------|-----------|----------------------|
| **검색 방식** | 임베딩 유사도 | Cypher 쿼리 (그래프 패턴 매칭) |
| **관계 표현** | ❌ 불가능 | ✅ 명시적 관계 저장 |
| **복잡한 질의** | ❌ 제한적 | ✅ 다단계 관계 추론 가능 |
| **구현 난이도** | ⭐⭐ 쉬움 | ⭐⭐⭐⭐ 어려움 |
| **정확성** | 의미적 유사도 기반 | 구조적 관계 기반 (더 정확) |
| **예시 질문** | "AI 관련 뉴스 찾아줘" | "뉴스1의 경제 기사를 쓴 기자는?" |

#### Vector RAG의 한계를 GraphRAG가 해결하는 예시

**질문**: "뉴스1에서 발행한 경제 뉴스를 작성한 기자는 누구야?"

- **Vector RAG**: 
  - 문제: "뉴스1", "경제", "기자"를 모두 포함한 문서를 찾기 어려움
  - 결과: 부정확하거나 불완전한 답변

- **GraphRAG**: 
  - 해결: `(기자)-[:WRITTEN_BY]-(뉴스)-[:PUBLISHED_BY]-(뉴스1)` 경로 탐색
  - 결과: 정확한 관계 기반 답변


### 7.4 실제 활용 시나리오

#### 1. 언론사 모니터링 시스템
```python
# 특정 언론사의 특정 카테고리 기사 트렌드 분석
"SBS에서 최근 IT과학 카테고리 기사를 많이 쓰는 기자는?"
```

#### 2. 기자 성과 분석 시스템
```python
# 기자별 생산성 및 전문 분야 파악
"가장 많은 경제 기사를 작성한 기자 3명과 그들의 최근 기사는?"
```

#### 3. 뉴스 추천 시스템
```python
# 사용자 관심사 기반 뉴스 추천
"자율주행 관련 뉴스를 주로 다루는 언론사와 기자를 추천해줘"
```

#### 4. 콘텐츠 큐레이션
```python
# 다양한 관점의 뉴스 수집
"같은 주제를 다루는 서로 다른 언론사의 기사를 비교해줘"
```


### 7.5 시스템 확장 가능성

#### 1. 새로운 쿼리 템플릿 추가
```python
class CypherQueryTemplates(Enum):
    # 기존 템플릿...
    
    # 새로운 템플릿 추가
    TRENDING_NEWS = "trending_news"
    
    def build(self, **kwargs):
        if self is CypherQueryTemplates.TRENDING_NEWS:
            return """
            MATCH (n:News)
            WHERE n.publishDate >= date() - duration('P7D')
            RETURN n.title, count(*) as views
            ORDER BY views DESC
            LIMIT 10
            """
```

#### 2. 추가 Tool 구현
```python
@tool("analyze_news_sentiment")
def analyze_news_sentiment(news_id: str) -> str:
    """뉴스 감성 분석 도구"""
    # 감성 분석 로직
    pass

@tool("find_related_news")
def find_related_news(news_id: str) -> str:
    """관련 뉴스 찾기 도구"""
    # 관련 뉴스 검색 로직
    pass
```

#### 3. 하이브리드 접근 (Vector + Graph)
```python
# Vector 검색으로 후보 뉴스 찾기
# → Graph 쿼리로 관련 엔티티 및 관계 탐색
# → 컨텍스트가 풍부한 답변 생성
```

#### 4. 실시간 스트리밍 응답
```python
# LangGraph의 stream() 메서드 활용
for chunk in agent.stream({"messages": [("user", question)]}):
    # 실시간으로 응답 출력
    print(chunk)
```


### 7.6 핵심 학습 포인트

#### ✅ 이 강의에서 배운 것

1. **GraphRAG의 개념과 필요성**
   - Vector RAG의 한계 (관계 표현 불가)
   - Graph 데이터베이스의 강점 (명시적 관계 표현)

2. **Neo4j 그래프 모델링**
   - 노드와 관계로 데이터 구조화
   - Cypher 쿼리 언어 활용

3. **CypherQueryTemplates (Enum 패턴)**
   - 쿼리를 템플릿화하여 재사용성 향상
   - 유지보수성과 타입 안정성 확보

4. **LangChain Tool 개발**
   - `@tool` 데코레이터로 함수를 LLM이 사용 가능한 도구로 변환
   - Pydantic 모델로 파라미터 타입 정의
   - 명확한 description으로 LLM의 도구 선택 가이드

5. **ReAct Agent 패턴**
   - Thought (추론) → Action (행동) → Observation (관찰) 사이클
   - 복잡한 문제를 단계별로 해결

6. **LangGraph 활용**
   - `create_react_agent`로 간단히 Agent 구축
   - 상태 관리 및 워크플로우 제어
   - 디버깅과 모니터링 용이

#### 🎯 이 시스템의 핵심 가치

- **정확성**: 구조화된 그래프 기반 검색으로 높은 정확도
- **유연성**: 새로운 쿼리 템플릿과 Tool 추가 용이
- **확장성**: 다양한 도메인에 적용 가능한 아키텍처
- **투명성**: ReAct 패턴으로 AI의 추론 과정 확인 가능

#### 💡 실무 적용 팁

1. **쿼리 템플릿 설계 시**
   - 자주 사용되는 패턴을 먼저 템플릿화
   - 파라미터는 최소화하되 유연성 확보
   - 성능을 고려한 인덱스 설계

2. **Tool 설명 작성 시**
   - 구체적인 사용 예시 포함
   - 입력 파라미터 설명을 명확하게
   - 어떤 상황에서 사용해야 하는지 명시

3. **System Prompt 작성 시**
   - Agent의 역할과 책임 명확히 정의
   - 사용 가능한 도구와 템플릿 목록 제공
   - 응답 스타일 가이드라인 제시

4. **에러 처리**
   - Tool 실행 실패 시 명확한 에러 메시지
   - 대안 제시 (다른 검색어, 다른 템플릿 등)
   - 로깅으로 문제 추적 가능하도록 설계


### 7.7 다음 단계 및 참고 자료

#### 🚀 추가 학습 방향

1. **Neo4j 고급 기능**
   - Graph Data Science (GDS) 라이브러리 활용
   - 커뮤니티 탐지, 중심성 분석 등
   - 그래프 임베딩과 머신러닝 통합

2. **하이브리드 RAG**
   - Vector 검색 + Graph 검색 결합
   - 의미적 유사도와 구조적 관계 모두 활용
   - 더 정확하고 풍부한 컨텍스트 제공

3. **멀티 에이전트 시스템**
   - 여러 Agent가 협업하는 시스템
   - 각 Agent가 특화된 역할 담당
   - 복잡한 문제를 분산하여 해결

4. **프로덕션 배포**
   - FastAPI로 REST API 구축
   - 캐싱 전략 (Redis 등)
   - 모니터링 및 로깅 (Prometheus, Grafana)

#### 📚 참고 자료

- **Neo4j**: https://neo4j.com/docs/
- **LangChain**: https://python.langchain.com/
- **LangGraph**: https://langchain-ai.github.io/langgraph/
- **ReAct Paper**: https://arxiv.org/abs/2210.03629
- **GraphRAG**: https://arxiv.org/abs/2404.16130

#### 🎓 마치며

이 강의를 통해 Neo4j와 LangGraph를 활용한 **GraphRAG 시스템**을 구축하는 방법을 배웠습니다.

**핵심 요약**:
- ✅ CypherQueryTemplates로 쿼리 템플릿화
- ✅ LangChain Tool로 Neo4j 연동
- ✅ ReAct Agent로 지능적인 추론 및 행동
- ✅ LangGraph로 복잡한 워크플로우 관리

이 시스템은 뉴스 검색뿐만 아니라 **소셜 네트워크 분석**, **추천 시스템**, **지식 그래프 QA** 등 다양한 도메인에 적용할 수 있습니다.

**실무에서 GraphRAG를 적용할 때**는:
1. 도메인 특성에 맞는 그래프 모델 설계
2. 자주 사용되는 쿼리 패턴 파악 및 템플릿화
3. 명확한 Tool 설명으로 Agent의 선택 가이드
4. 충분한 테스트와 모니터링

**Happy Coding! 🚀**


## 8. 리소스 정리


In [ ]:
# Neo4j 연결 종료
neo4j_conn.close()
